# VN30F1M Momentum Transformer (Colab)

This notebook mounts Google Drive, sets paths, updates config, and runs the full pipeline.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Fill these 3 values before running
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/MomentumTransformer/vn30f1m_momentum_transformer'
DRIVE_DATA_CSV = '/content/drive/MyDrive/MomentumTransformer/data/vn30f1m.csv'
WORKDIR = '/content/vn30f1m_momentum_transformer'

print(DRIVE_PROJECT_DIR)
print(DRIVE_DATA_CSV)
print(WORKDIR)

In [ ]:
import os
import shutil
from pathlib import Path

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
shutil.copytree(DRIVE_PROJECT_DIR, WORKDIR)

raw_dir = Path(WORKDIR) / 'data' / 'raw'
raw_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(DRIVE_DATA_CSV, raw_dir / 'vn30f1m.csv')

print('Project synced to:', WORKDIR)
print('CSV copied to:', raw_dir / 'vn30f1m.csv')

In [ ]:
%cd /content/vn30f1m_momentum_transformer
!python -m pip install -U pip
!pip install -q pandas numpy pyyaml pyarrow scikit-learn matplotlib torch

In [ ]:
from pathlib import Path
import yaml

cfg_path = Path('configs/default.yaml')
cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8'))

cfg['paths']['project_root'] = '/content/vn30f1m_momentum_transformer'
cfg['paths']['raw_data_csv'] = 'data/raw/vn30f1m.csv'
cfg['trading']['base_round_trip_cost_points'] = 0.20
cfg['trading']['cost_scenarios_points'] = [0.20]

cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print('Updated config:', cfg_path)

In [ ]:
!python src/data_preprocessing.py --config configs/default.yaml
!python src/feature_engineering.py --config configs/default.yaml

In [ ]:
# Fast smoke run first; increase later for full experiment
from pathlib import Path
import yaml

cfg_path = Path('configs/default.yaml')
cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
cfg.setdefault('walk_forward', {})['max_windows'] = 1
cfg.setdefault('training', {})['max_epochs_per_window'] = 1
cfg['training']['max_train_rows_per_window'] = 5000
cfg['training']['max_valid_rows_per_window'] = 2000
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print('Applied fast profile for smoke run')

In [ ]:
!python src/walk_forward.py --config configs/default.yaml
!python src/evaluate.py --config configs/default.yaml

In [ ]:
!ls -lah outputs/metrics
!ls -lah outputs/tables

In [ ]:
# Optional: sync outputs back to Drive
import shutil
from pathlib import Path

dst = Path('/content/drive/MyDrive/MomentumTransformer/outputs_colab')
dst.mkdir(parents=True, exist_ok=True)
shutil.copytree('outputs', dst / 'outputs_latest', dirs_exist_ok=True)
print('Saved outputs to:', dst / 'outputs_latest')